In [0]:
%pip install --quiet -U openai redlines mlflow pydentic llama-index==0.10.27 llama-index-llms-openai==0.1.15 llama-index-embeddings-openai==0.1.7
dbutils.library.restartPython()

In [0]:
from typing import Dict, List, Type, Union, Iterable, Optional, cast
from openai import OpenAI, NotGiven, NOT_GIVEN
from openai.lib._parsing import ResponseFormatT
from openai.types.chat_model import ChatModel
from openai.types.chat.chat_completion_message_param import ChatCompletionMessageParam
from openai.types.chat.chat_completion_tool_param import ChatCompletionToolParam

In [0]:
def get_client():
  base_url = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}/serving-endpoints'
  databricks_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

  client = OpenAI(
    api_key=databricks_token,
    base_url=base_url
  )

  return client


In [0]:
def get_completion_create(
  oai_client: OpenAI,
  model: Union[str, ChatModel],
  messages: Iterable[ChatCompletionMessageParam],
  tools = NOT_GIVEN,
  temperature: Optional[float] | NotGiven = NOT_GIVEN

):
  completion = oai_client.chat.completions.create(
    model = model,
    messages= messages,
    temperature = temperature,
    tools= tools
  )

  return completion

In [0]:
def get_completion_parse(
  oai_client: OpenAI,
  model: Union[str, ChatModel],
  messages: Iterable[ChatCompletionMessageParam],
  tools: Iterable[ChatCompletionToolParam] | NotGiven = NOT_GIVEN,
  response_format: type[ResponseFormatT] | NotGiven = NOT_GIVEN,
  temperature: Optional[float] | NotGiven = NOT_GIVEN

):
  completion = oai_client.beta.chat.completions.parse(
    model = model,
    messages = messages,
    temperature = temperature,
    tools = tools,
    response_format= response_format
  )

  return completion

In [0]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI as LlamaIndexOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import SummaryIndex, VectorStoreIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

In [0]:
def get_router_query_engine(file_path: str, llm = None, embed_model = None):
  """Get Router Quesry Engine"""
  llm= llm or LlamaIndexOpenAI(model="gpt-3.5-turbo")
  embed_model = embed_model or OpenAIEmbedding(model="text-embedding-ada-002")

  # laod documents
  documents = SimpleDirectoryReader(input_files=[file_path]).load_data()

  splitter = SentenceSplitter(chunk_size=1024)
  nodes = splitter.get_nodes_from_documents(documents)

  summary_index = SummaryIndex(nodes)
  vector_index = VectorStoreIndex(nodes, embed_model=embed_model)

  summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
    llm=llm
  )

  vector_query_engine = vector_index.as_query_engine(llm=llm)

  summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
      "Useful for summarization questions related to MetaGPT"
    ),
  )

  vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
      "Useful for retrieving specific context from the MetaGPT paper."
    ),
  )

  query_engine = RouterQueryEngine.from_defaults(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
      summary_tool,
      vector_tool
    ],
    verbose=True
  )
  return query_engine  

  

